# Bulk-crystal CXR via the Zhai Monte Carlo pipeline

Coherent X-ray (PXR + CBS) spectra and bremsstrahlung backgrounds from **bulk**
crystals under table-top electron energies, using the segment-sum Monte Carlo
of `cxr_montecarlo.py` (Zhai et al., Nat. Commun. 16, 11218 (2025) architecture):
CASINO-style transport (Browning/Mott elastic + Joy-Luo CSDA) -> per-segment
Eq. (13)/(14) amplitudes with finite-segment sinc$^2$ lineshapes -> per-segment
Beer-Lambert escape absorption -> EDS + aperture detector model.

**Crystals** (all with the relevant axis along the beam, normal incidence, bulk):

| crystal | reflections | geometry notes |
|---|---|---|
| HOPG graphite | $\pm(002), \pm(004)$ | fiber texture: only $(00l)$ coherent -- transport's amorphous assumption is *protected* |
| $\mathrm{MoSe}_2$ | $\pm(002), \pm(004), \pm(006)$ | c-axis $\parallel$ beam; basis verified: 4f Se at $z=0.621$ ($\delta=0.129$, matches the 2.53 $\mathrm{\AA}$ Mo-Se bond) |
| diamond | $\{111\}, \{022\}, \{004\}$ families | beam $\parallel$ [001] zone axis |
| silicon | $\{111\}, \{022\}, \{004\}$ families | beam $\parallel$ [001] zone axis |

**Caveats** (quantified in `kinematic_validity_check.py`):
- Single crystals at an exact zone axis: electron channeling ($\xi_e \sim$ mfp) is
  ignored by the amorphous transport; real measurements tilt a few degrees off axis.
  HOPG is exempt (no coherent transverse potential).
- B-factors approximate (graphite 0.8, $\mathrm{MoSe}_2$ 0.6 $\mathrm{\AA^2}$, unsourced).
- Bremsstrahlung: nonrelativistic Bethe-Heitler Born + Elwert, isotropic emission;
  for Mo/Se ($Z \gtrsim 30$) Seltzer-Berger tables would be better.
- Units are absolute -- **no calibration factors**: Phs/eV/s/nA into the EDS solid
  angle (0.066 sr at $\theta_{obs}=119°$, the Zhai experimental geometry).


In [ ]:
import time
from itertools import permutations, product

import numpy as np
import matplotlib.pyplot as plt

from cxr_feranchuk_spence import CRYSTALS
from cxr_montecarlo import (
    run_cases,
    beta_from_keV,
    eds_fwhm_eV,
    aperture_fwhm_eV,
    convolve_detector,
)

# ---- experiment DEFAULTS: every entry can be overridden per crystal ------------
# (just add the same key to a config in the next cell)
DEFAULTS = dict(
    theta_obs_deg=90.0,    # detector polar angle from the beam [deg]
    dtheta_obs_deg=5.0,    # detector polar-angle span (line smearing) [deg]
    domega_sr=0.066,       # detector solid angle [sr]
    thickness_ang=1e7,     # sample thickness [Ang]; 1e7 = 1 mm (bulk)
    tilt_deg=-30.0,        # sample-normal polar tilt [deg]; NEGATIVE tips the
                           # entrance face toward the detector. Must be nonzero
                           # (negative) at theta_obs = 90: an untilted infinite
                           # slab absorbs photons traveling parallel to its faces
    beam_uvw=(0, 0, 1),    # crystal axis along the slab normal ([001] = c-axis
                           # for hexagonal, cube edge for cubic)
)

ENERGIES_KEV = [30.0]      # beam energies (each crystal runs at each)
NE = 300                   # electrons per transport run (lines)
NE_BREM = 100              # electrons for the background (smooth, needs fewer)
PER_NA = 6.2415e9          # electrons/s at 1 nA


def family(hkl):
    """All sign/permutation members of a cubic reflection family."""
    out = set()
    for perm in permutations(hkl):
        for signs in product((1, -1), repeat=3):
            out.add(tuple(p * s for p, s in zip(perm, signs)))
    return sorted(out)


def pm(*hkls):
    """A list of reflections together with their negatives."""
    out = []
    for h in hkls:
        out += [tuple(h), tuple(-x for x in h)]
    return out


In [ ]:
# ---- per-crystal configuration ------------------------------------------------
# Required keys: crystal, composition, hkl_list, B_ang2, E_grid.
# Optional keys override DEFAULTS for that crystal only:
#   thickness_ang, beam_uvw, tilt_deg, theta_obs_deg, dtheta_obs_deg, domega_sr
def n_of(crystal, element):
    """Number density [1/Ang^3] of one element in a crystal's unit cell."""
    info = CRYSTALS[crystal]
    count = sum(1 for el, _ in info["basis"] if el == element)
    return count / info["V_cell"]


crystal_configs = {
    "graphite": dict(
        crystal="graphite",
        composition=[("C", n_of("graphite", "C"))],
        hkl_list=pm((0, 0, 2), (0, 0, 4)),
        B_ang2=0.8,
        E_grid=np.arange(400.0, 2400.0, 1.0),
    ),
    "mose2": dict(
        crystal="mose2",
        composition=[("Mo", n_of("mose2", "Mo")), ("Se", n_of("mose2", "Se"))],
        hkl_list=pm((0, 0, 2), (0, 0, 4), (0, 0, 6)),
        beam_uvw=(0, 0, 2),
        B_ang2=0.6,
        E_grid=np.arange(300.0, 2000.0, 1.0),
    ),
    "mose2_thin": dict(
        crystal="mose2",
        composition=[("Mo", n_of("mose2", "Mo")), ("Se", n_of("mose2", "Se"))],
        hkl_list=pm((0, 0, 2), (0, 0, 4), (0, 0, 6)),
        beam_uvw=(0, 0, 2),
        thickness_ang=1000.0,          # 100 nm film  <-- EDIT to taste
        B_ang2=0.6,
        E_grid=np.arange(300.0, 2000.0, 1.0),
    ),
    "diamond": dict(
        crystal="diamond",
        composition=[("C", n_of("diamond", "C"))],
        hkl_list=family((1, 1, 1)) + family((0, 2, 2)) + family((0, 0, 4)),
        beam_uvw=(4, 0, 0),
        B_ang2=0.21,
        E_grid=np.arange(500.0, 4300.0, 2.0),
    ),
    "silicon": dict(
        crystal="silicon",
        composition=[("Si", n_of("silicon", "Si"))],
        hkl_list=family((1, 1, 1)) + family((0, 2, 2)) + family((0, 0, 4)),
        beam_uvw=(4, 4, 0),
        B_ang2=0.46,
        E_grid=np.arange(500.0, 4000.0, 2.0),
    ),
}

# ---- build the case list (DEFAULTS merged with per-crystal overrides) ----------
cases = []
for i_c, (name, cfg) in enumerate(crystal_configs.items()):
    p = {**DEFAULTS, **cfg}
    for E0 in ENERGIES_KEV:
        cases.append(dict(
            name=name,
            crystal=p["crystal"],
            composition=p["composition"],
            hkl_list=p["hkl_list"],
            B_ang2=p["B_ang2"],
            E0_keV=E0,
            thickness_ang=p["thickness_ang"],
            E_grid=(p["E_grid"][0],
                    p["E_grid"][-1] + (p["E_grid"][1] - p["E_grid"][0]),
                    p["E_grid"][1] - p["E_grid"][0]),
            theta_obs_rad=np.deg2rad(p["theta_obs_deg"]),
            tilt_deg=p["tilt_deg"],
            beam_uvw=p["beam_uvw"],
            Ne=NE,
            Ne_brem=NE_BREM,
            seed=1000 * i_c + int(E0),
            # used downstream (detector model / unit scaling), ignored by run_case:
            dtheta_obs_rad=np.deg2rad(p["dtheta_obs_deg"]),
            domega_sr=p["domega_sr"],
        ))
    print(f"{name:11s}: {len(p['hkl_list']):2d} refl, "
          f"t = {p['thickness_ang']/1e4:8.3f} um, uvw = {p['beam_uvw']}, "
          f"tilt = {p['tilt_deg']:+5.1f} deg, theta_obs = {p['theta_obs_deg']:5.1f} deg, "
          f"dOmega = {p['domega_sr']:g} sr")


In [ ]:
# ---- transport + spectra (parallel) ---------------------------------------------
# Each (crystal, beam energy) case runs in its own PROCESS via
# cxr_montecarlo.run_cases (worker is module-level -> Windows-spawn safe).
# Two transports per case: E_cut = 5 keV for the lines, 1 keV for the
# continuum (coarse 10 eV grid, interpolated). max_workers=0 -> serial.
t_all = time.perf_counter()
outs = run_cases(cases, max_workers=None)
print(f"{len(cases)} cases in {time.perf_counter() - t_all:.0f} s (parallel)")

results = {}
for case, out in zip(cases, outs):
    name, E0 = case["name"], case["E0_keV"]
    E_grid = out["E_grid"]
    E_pk = E_grid[np.argmax(out["spec"])]
    # detector FWHM at this case's own geometry
    fwhm = np.sqrt(
        eds_fwhm_eV(E_pk) ** 2
        + aperture_fwhm_eV(E_pk, beta_from_keV(E0),
                           case["theta_obs_rad"], case["dtheta_obs_rad"]) ** 2
    )
    results.setdefault(name, {})[E0] = dict(
        E_grid=E_grid,
        spec=out["spec"],
        brem=out["brem"],
        E_pk=E_pk,
        fwhm=fwhm,
        eta=out["eta"],
        scale=case["domega_sr"] * PER_NA,   # (per e per sr) -> (per s per nA)
        case=case,
    )
    print(f"{name:11s} {E0:4.0f} keV: {out['n_segments']:6d} segments, "
          f"strongest line {E_pk:5.0f} eV, FWHM {fwhm:3.0f} eV")


In [ ]:
# ---- spectra: EDS-convolved lines + bremsstrahlung (dashed) --------------------
# NOTE: the detector convolution uses the FWHM at each case's strongest line;
# over wide grids the true FWHM varies ~sqrt(E) (few-% effect here).
n_cfg = len(crystal_configs)
ncols = 2
nrows = (n_cfg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), squeeze=False)
for ax in axes.ravel()[n_cfg:]:
    ax.axis("off")

for ax, name in zip(axes.ravel(), crystal_configs):
    for i, E0 in enumerate(ENERGIES_KEV):
        r = results[name][E0]
        line_det = convolve_detector(r["E_grid"], r["spec"], r["fwhm"])
        brem_det = convolve_detector(r["E_grid"], r["brem"], r["fwhm"])
        ax.plot(r["E_grid"] / 1e3, (line_det + brem_det) * r["scale"],
                color=f"C{i}", label=f"{E0:g} keV")
        ax.plot(r["E_grid"] / 1e3, brem_det * r["scale"], color=f"C{i}",
                ls="--", lw=1.0)
    c = r["case"]
    ax.set_title(f"{name}: t={c['thickness_ang']/1e4:g} um, uvw={c['beam_uvw']}, "
                 f"tilt={c['tilt_deg']:g}, "
                 f"$\\theta_o$={np.degrees(c['theta_obs_rad']):g}$\\degree$",
                 fontsize=10)
    ax.set_xlabel("Photon energy (keV)")
    ax.set_ylabel("Intensity (Phs/eV/s/nA)")
    ax.grid(alpha=0.3)
    ax.legend(title="solid: total, dashed: brem", fontsize=8)
fig.suptitle("Bulk/film CXR + bremsstrahlung, per-crystal geometry, EDS-convolved",
             fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
# ---- summary: peaks, ratios, detector count rates (unity QE) -------------------
print(f"{'config':11s} {'Ee':>5s} {'line':>6s} {'peak':>7s} {'pk/bg':>6s} "
      f"{'line cts':>9s} {'brem cts':>9s} {'total':>9s}   (band = full grid)")
print(f"{'':11s} {'keV':>5s} {'eV':>6s} {'Ph/eV/s/nA':>7s} {'':>6s} "
      f"{'cts/s/nA':>9s} {'cts/s/nA':>9s} {'cts/s/nA':>9s}")
print("-" * 92)
for name in crystal_configs:
    for E0 in ENERGIES_KEV:
        r = results[name][E0]
        line_det = convolve_detector(r["E_grid"], r["spec"], r["fwhm"])
        brem_det = convolve_detector(r["E_grid"], r["brem"], r["fwhm"])
        i_pk = np.argmax(line_det)
        line_cts = np.trapezoid(r["spec"], r["E_grid"]) * r["scale"]
        brem_cts = np.trapezoid(r["brem"], r["E_grid"]) * r["scale"]
        print(f"{name:11s} {E0:5.0f} {r['E_grid'][i_pk]:6.0f} "
              f"{line_det[i_pk] * r['scale']:7.2f} "
              f"{line_det[i_pk] / brem_det[i_pk]:6.2f} "
              f"{line_cts:9.1f} {brem_cts:9.1f} {line_cts + brem_cts:9.1f}")
print()
print("peak/bg: EDS-convolved peak height over the background at the peak;")
print("count rates assume unity quantum efficiency across the listed band.")
